# Run DynaGuide in the Calvin env

## Prerequisites

1. Install the env once with `setup_dynaguide.bat`, see [tutorial_for_pclab.md](../tutorial_for_pclab.md)
1. Download the DynaGuide model from this [link](https://drive.google.com/file/d/1DeOnoDacXjBHgy1DGoRJpYIR8SyDy5fJ/view?usp=drive_link)
1. Download the Base policy from this [link](https://drive.google.com/file/d/1lcI_PBgFIYDsoK4T4qO7SGJJgI2lw0Kd/view?usp=drive_link)
1. Download the Guidance conditions for SWITCH_ON from this [link](https://drive.google.com/file/d/1wtEGnG87Y-imqD2MygcqbJwp_RGi7YNB/view?usp=drive_link)
1. Put these three downloaded files in the folder _DynaGuide-main_, keeping their names. `calvin_exp_configs_examples/switch_on.json` already points to the guidance condition there, so nothing needs editing

## Running DynaGuide rollouts

Now, run this command in the windows, Anaconda Prompt. The `^` at the end of a line continues the command on the next line, like `\` on Linux. Nothing may follow it, not even a space, and the last line has none. While pasting, the prompt shows `More?` for each continued line, this is normal:
```bat
conda activate dynaguide
cd DynaGuide-main
python run_dynaguide.py ^
    --video_path results\output_video.mp4 ^
    --dataset_path results\output_rollout.hdf5 ^
    --dataset_obs ^
    --json_path results\output_log.json ^
    --horizon 400 ^
    --n_rollouts 3 ^
    --agent base_policy_300.pth ^
    --output_folder results ^
    --video_skip 2 ^
    --exp_setup_config calvin_exp_configs_examples\switch_on.json ^
    --guidance dynamics_model_5900.pth ^
    --camera_names third_person ^
    --scale 1.5 ^
    --ss 4 ^
    --alpha 30 ^
    --save_frames
```

The paths are relative, so it has to run from inside `DynaGuide-main`. A PyBullet window opens during the rollouts, leave it open. Results land in `DynaGuide-main/results`.

## Evaluating success automatically

CALVIN has no success signal of its own, so `core/calvin_behaviors.py` detects what the robot achieved from the states and compares this to the goal in the experiment config. You don't need a separate analysis step or config. After the run you can watch the video and read the result in the terminal and in `results\output_log.json`:

```json
"Target_Behaviors": ["switch_on"],
"Avoid_Behaviors": [],
"Success_Rate": 0.6667,
"Num_Success": 2,
"Num_Episode": 3,
"Behavior_Counts": {"switch_on": 2, "door_left": 1},
"Episode_Behaviors": [["switch_on"], ["door_left"], ["switch_on"]]
```

- `Success_Rate`, `Num_Success`: the rollouts that reached the goal. Both are `null` if the config has no goal, like `control.json`
- `Behavior_Counts`: how often each behavior happened. This also tells you what the robot did instead when it failed
- `Episode_Behaviors`: the behavior of each rollout, in order, so you can look up a specific rollout in the video

### Scoring a file you already have

This needs no rollouts, but the file must be saved with `--dataset_path` and `--dataset_obs`. Files from `run_trained_agent.py`, `run_sampling_baseline.py` and older `run_dynaguide.py` runs stop at the first small change, usually before the goal is reached, so they mostly score `no_behavior`:

```bat
python evaluate_behaviors.py ^
    --dataset results\output_rollout.hdf5 ^
    --config calvin_exp_configs_examples\switch_on.json ^
    --output results\behaviors.json
```

### Running other tasks

Pass a different config with `--exp_setup_config` and download the matching guidance condition. All configs in `calvin_exp_configs_examples` already have their goal, so only `pos_examples` / `neg_examples` need to point to your files. A goal looks like this:

```json
"target_behaviors" : ["switch_on"],
```

| Field | Meaning | Example config |
|---|---|---|
| `target_behaviors` | Success if **any** of these happens | `switch_on.json`, `multiple_and_negative_objectives/button_on_switch_on.json` |
| `avoid_behaviors` | Failure if any of these happens | `door_dont_right.json` |
| both | A target happened and nothing to avoid happened | |
| neither | Only the behavior counts are reported | `control.json` |

With only `avoid_behaviors`, a rollout where the robot does nothing counts as a success, so check `Behavior_Counts` for `no_behavior`. A typo in a behavior name stops the run with an error before the rollouts start.

### When a rollout ends

The config decides this, `run_dynaguide.py` only asks `GoalTracker` in `core/calvin_behaviors.py` whether the outcome is decided. A rollout ends when one of these happens:

- A behavior from `target_behaviors` is reached (success)
- A behavior from `avoid_behaviors` happens (failure)
- `--horizon` steps have passed. This is the only way to end for `control.json`, and for avoid-only configs it means success

Doing something else does not end the rollout: if the robot opens the drawer on the way to the switch, it continues and can still reach the switch. `Episode_Behaviors` lists everything it achieved, in order. So set `--horizon` high enough for the task, and expect failed rollouts to take the full time.

These results aren't comparable to the paper's numbers. The paper stops at the first small change and measures which object was touched first, while here the robot keeps trying until it reaches the goal. For a baseline, run the same config with `--scale 0 --ss 1`.

### Behaviors and thresholds

A behavior counts when the state goes from the start range to the end range, measured from the first state of the rollout. These are the end states from `data_processing_calvin/calvin_to_labeled_hdf5.py`, the authors' definition of a behavior in the guidance data:

| Behavior | Start | End | Range in the simulator |
|---|---|---|---|
| `door_left` / `door_right` | below 0.05 / above 0.25 | above 0.25 / below 0.05 | 0 to 0.27 |
| `drawer_open` / `drawer_close` | below 0.05 / above 0.10 | above 0.10 / below 0.05 | 0 to 0.16 |
| `switch_on` / `switch_off` | below 0.01 / above 0.07 | above 0.07 / below 0.01 | 0 to 0.09, the light changes at about 0.045 |
| `button_on` / `button_off` | below 0.2 / above 0.8 | above 0.8 / below 0.2 | 0 or 1 |
| `red_displace`, `blue_displace`, `pink_displace` | anywhere | moved more than 0.01 with the gripper within 0.1 | |
| `no_behavior` | | none of these happened | |

The block condition is the weakest one, 0.01 is only a small push. The block labels mean the block was moved, not necessarily lifted, even though the configs are called `red_lift` etc.

To make a goal stricter, change its numbers in `ARTICULATED` or `BLOCK_MOVED` in `core/calvin_behaviors.py`. Evaluation and termination both follow, because they use the same code. To add a new behavior, add its name to `BEHAVIORS` and its check to `GoalTracker.detect`. Then it can be used in any config.

The cell below shows the results of the last run. Run it from a kernel started in `DynaGuide-main`.

In [ ]:
import json

with open("results/output_log.json", "r") as f:
    results = json.load(f)

print(f"Goal: {results['Target_Behaviors']}, avoid: {results['Avoid_Behaviors']}")
print(f"Success: {results['Num_Success']} / {results['Num_Episode']}")
print(f"Behaviors: {results['Behavior_Counts']}")
for i, behaviors in enumerate(results["Episode_Behaviors"]):
    print(f"Rollout {i}: {behaviors}")

To check the guidance conditions, run:
```bat
python helper_scripts\show_guidance_images.py ^
    --dataset guidance_condition_switch_on_20.hdf5 ^
    --output results\guidance_images
```